# Demo D1. The forced heat equation

**Partial differential equations · diffusion.**

The heat equation $u_t = u_{xx} + f(x)$ on $x \in [0, 1]$ models the temperature in a rod whose ends are held at fixed values $u(0,t)=a$ and $u(1,t)=b$ (Dirichlet boundary conditions), with a steady heat source $f(x)$. Diffusion smooths the profile while the source feeds it; over time the temperature relaxes to a **steady state** $v(x)$ that no longer changes. This lab marches the equation forward with an explicit finite-difference scheme, watches the profile evolve from an initial condition $u(x,0)$, and compares it against the steady state it approaches.

In [ ]:
%pip install ipywidgets

In [ ]:
# --- Colab / Jupyter setup ------------------------------------------------
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3d projection)
from ipywidgets import interact, Dropdown, FloatSlider

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)

## Discretization

Put a grid $x_i = i\,\Delta x$, $i = 0, \dots, N$, with $\Delta x = 1/N$. Approximating $u_{xx}$ by the centered second difference gives the explicit (forward-time, centered-space) update
$$ u_i^{n+1} = u_i^n + r\,(u_{i+1}^n - 2u_i^n + u_{i-1}^n) + \Delta t\, f(x_i), \qquad r = \frac{\Delta t}{\Delta x^2}. $$
This scheme is stable only when $r \le \tfrac{1}{2}$; we fix $r = 0.4$ and pick $\Delta t$ accordingly. The boundary values $u_0 = a$ and $u_N = b$ are reset at every step.

## Steady state

At equilibrium $u_t = 0$, so the steady profile solves the two-point boundary-value problem $-v''(x) = f(x)$ with $v(0)=a$, $v(1)=b$. With no source it is simply the straight line from $a$ to $b$; a source bends it. Discretizing $v''$ by the same centered difference turns this into a small tridiagonal linear system, solved once below.

In [ ]:
# ---------------------------------------------------------------------------
# Spatial grid, and the two ingredients the user chooses: a source f(x) and an
# initial profile u(x,0). Kept as small catalogs so the dropdowns stay readable;
# add an entry to either dict to extend the lab.
# ---------------------------------------------------------------------------
N  = 60                          # number of grid intervals
x  = np.linspace(0.0, 1.0, N + 1)
dx = x[1] - x[0]
r  = 0.4                         # diffusion number dt/dx^2; must stay < 0.5
dt = r * dx**2

FORCINGS = {
    "none":               lambda x: np.zeros_like(x),
    "constant heat":      lambda x: 60.0 * np.ones_like(x),
    "sine source":        lambda x: 150.0 * np.sin(np.pi * x),
    "localized (center)": lambda x: 400.0 * np.exp(-100.0 * (x - 0.5) ** 2),
    "dipole (+/-)":       lambda x: 400.0 * (np.exp(-120 * (x - 0.3) ** 2)
                                             - np.exp(-120 * (x - 0.7) ** 2)),
}

INITIALS = {
    "zero":            lambda x: np.zeros_like(x),
    "sine bump":       lambda x: 8.0 * np.sin(np.pi * x),
    "triangle":        lambda x: 8.0 * np.where(x < 0.5, 2 * x, 2 * (1 - x)),
    "off-center step": lambda x: np.where(x < 0.5, 8.0, 0.0),
}

In [ ]:
# ---------------------------------------------------------------------------
# Steady state: solve -v'' = f on [0,1] with v(0)=a, v(1)=b by the centered
# difference, i.e. a tridiagonal linear system for the N-1 interior values.
# ---------------------------------------------------------------------------
def steady_state(f, a, b):
    fi = f(x)[1:-1]
    n = N - 1
    A = (np.diag(-2.0 * np.ones(n))
         + np.diag(np.ones(n - 1), 1)
         + np.diag(np.ones(n - 1), -1)) / dx**2
    rhs = -fi.copy()
    rhs[0]  -= a / dx**2         # move the known boundary values to the RHS
    rhs[-1] -= b / dx**2
    v_interior = np.linalg.solve(A, rhs)
    return np.concatenate(([a], v_interior, [b]))


def evolve(f_arr, u0, a, b, T):
    """March the explicit scheme from u0 to time T and return the final profile."""
    u = u0.copy()
    u[0], u[-1] = a, b
    for _ in range(max(1, int(round(T / dt)))):
        u_new = u.copy()
        u_new[1:-1] = (u[1:-1] + r * (u[2:] - 2 * u[1:-1] + u[:-2])
                       + dt * f_arr[1:-1])
        u_new[0], u_new[-1] = a, b
        u = u_new
    return u

## Watch it relax

Drag the **time** slider: the profile starts at the initial condition (grey dashed) and bends toward the steady state (red dashed) set by the source and the boundary values.

In [ ]:
def show_snapshot(forcing="sine source", initial="zero", a=0.0, b=0.0, T=0.05):
    f  = FORCINGS[forcing]
    u0 = INITIALS[initial](x)
    v  = steady_state(f, a, b)
    u  = evolve(f(x), u0, a, b, T)

    plt.figure()
    plt.plot(x, u0, color="0.7", ls="--", label="initial $u(x,0)$")
    plt.plot(x, v, "r--", lw=1.5, label="steady state $v(x)$")
    plt.plot(x, u, "b-", lw=2.5, label=rf"$u(x,\,t={T:.3f})$")
    plt.scatter([0, 1], [a, b], color="k", zorder=5)
    plt.xlabel("x"); plt.ylabel("temperature $u$")
    plt.title("Forced heat equation: relaxation toward steady state")
    plt.legend(loc="upper right"); plt.show()

show_snapshot()

In [ ]:
interact(
    show_snapshot,
    forcing=Dropdown(options=list(FORCINGS), value="sine source", description="f(x)"),
    initial=Dropdown(options=list(INITIALS), value="zero", description="u(x,0)"),
    a=FloatSlider(value=0.0, min=-10, max=10, step=0.5, description="u(0,t)=a"),
    b=FloatSlider(value=0.0, min=-10, max=10, step=0.5, description="u(1,t)=b"),
    T=FloatSlider(value=0.05, min=0.0, max=0.4, step=0.005, description="time t"),
);

## The whole evolution at once

Stacking the profile at successive times gives a surface $u(x,t)$. The front edge ($t=0$) is the initial condition; as $t$ grows the surface flattens onto the steady state.

In [ ]:
def show_surface(forcing="localized (center)", a=0.0, b=0.0, T=0.15, frames=60):
    fa = FORCINGS[forcing](x)
    u = INITIALS["zero"](x); u[0], u[-1] = a, b
    nsteps = max(1, int(round(T / dt)))
    rec_every = max(1, nsteps // frames)

    ts, U = [0.0], [u.copy()]
    for n in range(1, nsteps + 1):
        u_new = u.copy()
        u_new[1:-1] = (u[1:-1] + r * (u[2:] - 2 * u[1:-1] + u[:-2])
                       + dt * fa[1:-1])
        u_new[0], u_new[-1] = a, b
        u = u_new
        if n % rec_every == 0:
            ts.append(n * dt); U.append(u.copy())

    X, Tg = np.meshgrid(x, np.array(ts))
    fig = plt.figure(figsize=(9, 6))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(X, Tg, np.array(U), cmap="viridis")
    ax.set_xlabel("position x"); ax.set_ylabel("time t"); ax.set_zlabel("temp u")
    ax.set_title("Temperature evolution $u(x,t)$")
    plt.show()

interact(
    show_surface,
    forcing=Dropdown(options=list(FORCINGS), value="localized (center)", description="f(x)"),
    a=FloatSlider(value=0.0, min=-10, max=10, step=0.5, description="u(0,t)=a"),
    b=FloatSlider(value=0.0, min=-10, max=10, step=0.5, description="u(1,t)=b"),
    T=FloatSlider(value=0.15, min=0.02, max=0.4, step=0.01, description="max time"),
);

## Things to try

- With **none** as the source, the steady state is the straight line from $a$ to $b$: pure diffusion erases any initial bump.
- Turn on a source and watch the steady state bend away from that line; its shape solves $-v'' = f$.
- A **localized** source makes a sharp interior peak; the **dipole** heats one side and cools the other.
- Edit the cell to set $r > \tfrac{1}{2}$ and rerun: the explicit scheme blows up, showing the stability limit directly.

## Summary

- The forced heat equation relaxes any initial profile toward the steady state solving $-v'' = f$ with the given boundary values.
- The explicit scheme is simple but only conditionally stable: $r = \Delta t / \Delta x^2 \le \tfrac{1}{2}$.
- The source $f$ sets the equilibrium shape; the boundary values set its endpoints.